# Automatron Real Estate

Due-diligence red flags, permit pre-screens, and dispute summaries.

## Contents

1. Constants and thresholds
2. Sample data
3. Due-diligence tools
4. Permit tools
5. Dispute tools
6. Prompt addenda and workflows
7. Sector pack

Build the core module and put the generated package on the path. This cell is
for interactive use only and is dropped from the built module.

In [ ]:
import pathlib
import subprocess
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
subprocess.run([sys.executable, "scripts/build_notebooks.py"], check=True, cwd=ROOT)
sys.path.insert(0, str(ROOT / "automatron_build"))

In [ ]:
from automatron_core import *  # noqa: F401,F403

## 1. Constants and thresholds

Rule file loaders, page splitting, and date parsing.

In [ ]:
import datetime as dt
import functools
import json
import pathlib
import re
from typing import Any

import yaml

SECTOR_ID = "realestate"
# This sector has no numeric thresholds in the sector config: every limit it applies
# comes from a rules file that also carries the section it came from.
THRESHOLDS = sector_settings(SECTOR_ID).get("thresholds") or {}

SAMPLE_DIR = get_settings().data_path / "samples" / SECTOR_ID
RULES_DIR = ROOT / "config" / "rules"

# Pages are marked in the sample documents so every extracted clause can cite one.
# Real packets arrive as PDFs; the page marker is what a text extractor would give.
PAGE_MARKER = re.compile(r"^=== PAGE (\d+) ===\s*$")

TODAY = dt.date(2026, 5, 4)

SAMPLE_GENERATOR_VERSION = 1


@functools.lru_cache(maxsize=2)
def load_dd_severity() -> dict[str, Any]:
    with (RULES_DIR / "dd_severity.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_zoning() -> dict[str, Any]:
    with (RULES_DIR / "zoning_sample.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_notice_periods() -> dict[str, Any]:
    with (RULES_DIR / "notice_periods.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def split_pages(text: str) -> list[tuple[int, str]]:
    """Split a document into (page number, text) pairs on its page markers.

    Everything before the first marker is page one, so a document with no markers
    still cites a page rather than nothing.
    """
    pages: list[tuple[int, list[str]]] = []
    current = 1
    buffer: list[str] = []
    for line in text.splitlines():
        marker = PAGE_MARKER.match(line)
        if marker:
            if buffer:
                pages.append((current, buffer))
                buffer = []
            current = int(marker.group(1))
            continue
        buffer.append(line)
    if buffer:
        pages.append((current, buffer))

    # Anything before the first marker is page one, which means a document whose
    # preamble sits above its own "PAGE 1" marker would otherwise yield page one
    # twice and report the same clause against both.
    merged: dict[int, list[str]] = {}
    for number, lines in pages:
        merged.setdefault(number, []).extend(lines)
    return [(number, "\n".join(lines).strip()) for number, lines in sorted(merged.items())]


def _date(value: str) -> dt.date | None:
    for pattern in ("%Y-%m-%d", "%d %B %Y", "%B %d, %Y", "%d/%m/%Y", "%m/%d/%Y"):
        try:
            return dt.datetime.strptime(value.strip(), pattern).date()
        except (TypeError, ValueError):
            continue
    return None

## 2. Sample data

ensure_samples(): invented title, survey, association,
inspection, environmental and lease documents, plus projects and disputes.

In [ ]:
# Every document, property, party and project below is invented. The addresses,
# names, section numbers and recording references are fictional, and the standards
# they are measured against belong to a fictional city. Nothing here reproduces a
# real title commitment, survey, lease or code.

SYNTHETIC_NOTE = ("Invented documents written for this project. Not a real title "
                  "commitment, survey, lease, inspection report or municipal code.")

TITLE_COMMITMENT = """=== PAGE 1 ===
SAMPLE TITLE INSURANCE COMPANY (fictional)
COMMITMENT FOR TITLE INSURANCE
File No. STC-2026-00418
Property: 118 Alder Lane, Sample City (fictional)
Effective Date: 2026-04-02
Proposed Insured: Sample Buyer LLC (fictional)

This commitment is an invented document prepared to illustrate a review workflow.

=== PAGE 2 ===
SCHEDULE A
1. Policy to be issued: Owner's Policy, amount 615,000.00
2. Estate or interest: Fee simple
3. Title is vested in: Sample Seller Trust (fictional)
4. The land is described in Exhibit A.

=== PAGE 3 ===
SCHEDULE B - PART I - REQUIREMENTS
1. Pay the agreed amounts for the interest to be insured.
2. Release of the unreleased mortgage recorded in Book 4412, Page 208, in favor of
   Harbour Savings Bank (fictional), original amount 214,000.00. No satisfaction of
   this mortgage appears of record.
3. Payment of all taxes due and payable.

=== PAGE 4 ===
SCHEDULE B - PART II - EXCEPTIONS
1. A ten (10) foot wide utility easement along the entire rear property line,
   recorded in Book 2201, Page 77, in favor of Sample City Utilities (fictional),
   for the installation and maintenance of underground utilities.
2. Covenants, conditions and restrictions recorded in Book 3010, Page 145.
3. General exception: any encroachment, encumbrance, violation or adverse
   circumstance that an accurate and complete land survey would disclose.
4. Rights of tenants in possession under any unrecorded lease.

=== PAGE 5 ===
EXHIBIT A - LEGAL DESCRIPTION
Lot 14, Block 6, Alderwood Addition to Sample City (fictional), according to the
plat recorded in Plat Book 12, Page 3.
"""

SURVEY_NOTES = """=== PAGE 1 ===
SAMPLE LAND SURVEYING (fictional)
BOUNDARY SURVEY NOTES
Property: 118 Alder Lane, Sample City (fictional)
Field date: 2026-03-27
Surveyor note: this is an invented document.

=== PAGE 2 ===
OBSERVATIONS
1. The wood fence along the south boundary sits approximately 1.4 feet inside the
   adjoining parcel at the rear corner. This is an encroachment of the fence onto
   the neighbouring lot.
2. A ten foot utility easement is shown along the rear property line, consistent
   with Book 2201, Page 77.
3. The rear yard measures approximately 34 feet from the dwelling to the rear
   property line.
4. No structures were observed within the easement area at the time of survey.
"""

CCR = """=== PAGE 1 ===
ALDERWOOD HOMEOWNERS ASSOCIATION (fictional)
DECLARATION OF COVENANTS, CONDITIONS AND RESTRICTIONS
Recorded in Book 3010, Page 145
An invented document written to illustrate a review workflow.

=== PAGE 2 ===
ARTICLE IV - ARCHITECTURAL CONTROL
4.1 No building, fence, wall, detached garage, shed or other accessory structure
    shall be erected, placed or altered on any lot until the plans have been
    approved in writing by the Architectural Review Committee.
4.2 No accessory structure shall exceed 200 square feet in footprint without the
    written consent of the Committee.
4.3 Detached accessory structures are restricted in the rear yard setback area and
    shall not be placed within any recorded easement.

=== PAGE 3 ===
ARTICLE VI - USE RESTRICTIONS
6.1 Lots shall be used for single family residential purposes only.
6.2 No commercial activity shall be conducted on any lot other than a home
    occupation permitted by the applicable zoning ordinance.
"""

INSPECTION_REPORT = """=== PAGE 1 ===
SAMPLE HOME INSPECTIONS (fictional)
INSPECTION REPORT
Property: 118 Alder Lane, Sample City (fictional)
Inspection date: 2026-04-08
This is an invented report.

=== PAGE 2 ===
ROOF
The asphalt shingle roof shows granule loss and two areas of cupping on the south
slope. Estimated remaining service life is three to five years. This is a major
system item and further evaluation by a roofing contractor is recommended.

HVAC
The furnace is a 2004 unit at the end of its typical service life. The heat
exchanger could not be fully inspected. This is a major system item.

=== PAGE 3 ===
ELECTRICAL
The panel is modern and no deficiencies were observed.

INTERIOR
A cracked tile was noted in the upstairs bathroom. This is a minor cosmetic item.

EXTERIOR
The rear gate latch is broken. Minor item.
"""

ENVIRONMENTAL_SUMMARY = """=== PAGE 1 ===
SAMPLE ENVIRONMENTAL SERVICES (fictional)
PHASE I ENVIRONMENTAL SITE ASSESSMENT SUMMARY
Property: 118 Alder Lane, Sample City (fictional)
Report date: 2026-03-30
An invented summary written for this project.

=== PAGE 2 ===
FINDINGS
No recognized environmental conditions were identified on the subject property.
A former dry cleaner operated at 204 Alder Lane, approximately 180 feet upgradient
of the subject property, between 1978 and 1994. The assessor recommends further
review of that adjoining use before relying on this summary.
"""

LEASE = """=== PAGE 1 ===
RESIDENTIAL LEASE AGREEMENT (fictional)
Premises: 118 Alder Lane, Unit B, Sample City (fictional)
Landlord: Sample Seller Trust (fictional)
Tenant: Sample Tenant (fictional)
An invented lease written for this project.

=== PAGE 2 ===
2. TERM
The term begins 2025-09-01 and ends 2026-08-31.

3. RENT
Rent is 1,850.00 per month, due on the first day of each month.

4. SECURITY DEPOSIT
Tenant has paid a security deposit of 1,850.00.

=== PAGE 3 ===
9. ASSIGNMENT
Tenant shall not assign this lease or sublet the premises without the prior written
consent of Landlord. Any change of control of Tenant is deemed an assignment.

11. RENEWAL
This lease renews automatically for successive one year terms unless either party
gives written notice at least sixty (60) days before the end of the term.

12. QUIET ENJOYMENT
Tenant shall not create noise that unreasonably disturbs other residents.
"""

DOCUMENTS = {
    "title_commitment": TITLE_COMMITMENT,
    "survey_notes": SURVEY_NOTES,
    "ccr": CCR,
    "inspection_report": INSPECTION_REPORT,
    "environmental_summary": ENVIRONMENTAL_SUMMARY,
    "lease": LEASE,
}

PROJECTS = {
    # Every standard satisfied with room to spare.
    "project_r1_compliant": {
        "project_name": "Alder Lane single family (fictional)",
        "zone_district": "R-1", "use": "single_family_dwelling",
        "lot_area_sqft": 8000, "lot_width_ft": 70,
        "front_setback_ft": 28, "side_setback_ft": 9, "rear_setback_ft": 25,
        "height_ft": 26, "stories": 2,
        "gross_floor_area_sqft": 3600, "building_footprint_sqft": 2400,
        "dwelling_units": 1, "parking_spaces": 2,
    },
    # One genuine violation, and four standards sitting exactly on their limit, which
    # must pass: an off-by-one comparison here would fail a compliant project.
    "project_r1_setback_violation": {
        "project_name": "Birch Street addition (fictional)",
        "zone_district": "R-1", "use": "single_family_dwelling",
        "lot_area_sqft": 6500, "lot_width_ft": 62,
        "front_setback_ft": 25, "side_setback_ft": 5, "rear_setback_ft": 20,
        "height_ft": 30, "stories": 2,
        "gross_floor_area_sqft": 3250, "building_footprint_sqft": 2600,
        "dwelling_units": 1, "parking_spaces": 2,
    },
    # Dimensionally fine, but the use needs a hearing in this district.
    "project_c1_conditional_use": {
        "project_name": "Cedar Row cafe (fictional)",
        "zone_district": "C-1", "use": "restaurant",
        "lot_area_sqft": 5200, "lot_width_ft": 45,
        "front_setback_ft": 0, "side_setback_ft": 0, "rear_setback_ft": 12,
        "height_ft": 32, "stories": 2,
        "gross_floor_area_sqft": 6000, "building_footprint_sqft": 3000,
        "dwelling_units": 1, "parking_spaces": 6,
    },
}

PLAN_TEXT = """=== PAGE 1 ===
SITE PLAN NOTES (fictional)
Project: Birch Street addition
Zone district: R-1
Lot area: 6500 sq ft
Lot width: 62 ft
Front setback: 25 ft
Side setback: 6 ft
Rear setback: 20 ft
Building height: 30 ft
Stories: 2
Gross floor area: 3250 sq ft
Building footprint: 2600 sq ft
"""

DISPUTES = {
    "dispute_hoa_fence": {
        "dispute_type": "hoa_rule",
        "jurisdiction": "Sample City (fictional)",
        "parties": [
            {"name": "Sample Owner", "role": "owner"},
            {"name": "Alderwood HOA (fictional)", "role": "association"},
        ],
        "complaint_text": (
            "The association says my rear fence was built without architectural approval "
            "and has asked me to remove it. I submitted plans by email before building and "
            "did not hear back for six weeks, so I went ahead."
        ),
        "correspondence": (
            "2026-02-10 - Owner to HOA: Submitting plans for a 6 foot rear fence for "
            "architectural review.\n"
            "2026-03-24 - HOA to Owner: Notice of violation. The fence at the rear of the "
            "property was installed without written approval as required by Article IV.\n"
            "2026-03-28 - Owner to HOA: I submitted plans on 10 February and received no "
            "response. I am requesting a hearing.\n"
            "2026-04-02 - HOA to Owner: Your hearing request is acknowledged. The committee "
            "will consider this at its next meeting.\n"
        ),
        "documents": ["ccr"],
    },
    "dispute_deposit": {
        "dispute_type": "deposit",
        "jurisdiction": "Sample City (fictional)",
        "parties": [
            {"name": "Sample Tenant", "role": "tenant"},
            {"name": "Sample Seller Trust (fictional)", "role": "landlord"},
        ],
        "complaint_text": (
            "I moved out on 1 March and have not received my deposit or an itemised "
            "statement. The landlord says there was carpet damage but has not sent "
            "photographs or receipts."
        ),
        "correspondence": (
            "2026-03-01 - Tenant to Landlord: Keys returned, forwarding address provided.\n"
            "2026-03-20 - Tenant to Landlord: Following up on the security deposit.\n"
            "2026-03-27 - Landlord to Tenant: There was damage to the carpet in the second "
            "bedroom. We are still obtaining a quote.\n"
            "2026-04-14 - Tenant to Landlord: It has now been six weeks. Please send an "
            "itemised statement or return the deposit. I am considering small claims court.\n"
        ),
        "documents": ["lease"],
    },
    "dispute_noise_escalating": {
        "dispute_type": "noise",
        "jurisdiction": "Sample City (fictional)",
        "parties": [
            {"name": "Sample Tenant", "role": "tenant"},
            {"name": "Sample Manager", "role": "property_manager"},
        ],
        "complaint_text": (
            "The manager has sent increasingly hostile messages about noise from my unit "
            "and has threatened to change the locks and shut off the water if I do not move "
            "out."
        ),
        "correspondence": (
            "2026-04-02 - Manager to Tenant: We have received noise complaints about your "
            "unit after 11pm. Please keep noise down.\n"
            "2026-04-09 - Manager to Tenant: Final notice. This is your last warning.\n"
            "2026-04-18 - Manager to Tenant: If you do not move out by the end of the month "
            "I will change the locks and shut off the water. People like you always cause "
            "trouble in this building.\n"
            "2026-04-19 - Tenant to Manager: I am asking you to put any notice in writing "
            "through the proper process. I have contacted a lawyer.\n"
        ),
        "documents": ["lease"],
    },
}


def ensure_samples(force: bool = False) -> None:
    """Write the bundled real estate samples if they are missing or out of date."""
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    stamp = SAMPLE_DIR / ".generator_version"
    current = stamp.read_text(encoding="utf-8").strip() if stamp.is_file() else ""
    if not force and current == str(SAMPLE_GENERATOR_VERSION) and \
            (SAMPLE_DIR / "title_commitment.txt").is_file():
        return

    docs_note = f"[{SYNTHETIC_NOTE}]\n\n"
    for name, body in DOCUMENTS.items():
        (SAMPLE_DIR / f"{name}.txt").write_text(docs_note + body, encoding="utf-8")
    (SAMPLE_DIR / "plan_notes.txt").write_text(docs_note + PLAN_TEXT, encoding="utf-8")

    for name, project in PROJECTS.items():
        payload = {**project, "note": SYNTHETIC_NOTE}
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(payload, indent=2) + "\n", encoding="utf-8")

    for name, dispute in DISPUTES.items():
        payload = {**dispute, "note": SYNTHETIC_NOTE}
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(payload, indent=2) + "\n", encoding="utf-8")

    packets = {
        "packet_purchase": {
            "property_address": "118 Alder Lane, Sample City (fictional)",
            "deal_type": "purchase",
            "intended_use": "build a detached garage in the rear yard",
            "documents": ["title_commitment", "survey_notes", "ccr", "inspection_report",
                          "environmental_summary"],
        },
        "packet_lease": {
            "property_address": "118 Alder Lane, Unit B, Sample City (fictional)",
            "deal_type": "lease",
            "intended_use": "operate a small home bakery from the unit",
            "documents": ["lease", "ccr"],
        },
    }
    for name, packet in packets.items():
        packet["note"] = SYNTHETIC_NOTE
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(packet, indent=2) + "\n", encoding="utf-8")

    stamp.write_text(str(SAMPLE_GENERATOR_VERSION) + "\n", encoding="utf-8")

## 3. Due-diligence tools

Document identification, clause extraction with page
references, the intended-use cross-check, the severity rubric, and a draft
question list for counsel.

In [ ]:
# Keywords that identify a document type, and the clause categories each phrase
# points at. Matching on wording is crude, which is why every extracted item carries
# the page and the sentence it came from: the reader checks the source, not the label.
DOCUMENT_SIGNATURES = {
    "title_commitment": ("commitment for title insurance", "schedule b", "schedule a",
                         "requirements", "exceptions", "policy to be issued"),
    "survey_notes": ("boundary survey", "surveyor", "field date", "plat"),
    "ccr": ("covenants, conditions and restrictions", "declaration of covenants",
            "architectural review", "homeowners association"),
    "inspection_report": ("inspection report", "inspection date", "service life",
                          "hvac", "roof"),
    "environmental_summary": ("environmental site assessment", "phase i",
                              "recognized environmental conditions"),
    "lease": ("lease agreement", "landlord", "tenant", "security deposit", "rent is"),
}

CLAUSE_SIGNATURES = {
    "lien_encumbrance": ("mortgage", "lien", "judgment", "unreleased", "satisfaction of",
                         "deed of trust"),
    "easement": ("easement", "right of way", "right-of-way"),
    "restriction_ccr": ("covenant", "restriction", "shall not be erected",
                        "architectural review", "accessory structure", "residential purposes only"),
    "encroachment": ("encroach", "sits approximately", "inside the adjoining",
                     "over the boundary", "crosses the property line"),
    "title_exception": ("exception", "general exception", "rights of tenants"),
    "zoning_permit_note": ("nonconforming", "without a permit", "unpermitted", "zoning ordinance",
                           "home occupation"),
    "environmental": ("environmental", "dry cleaner", "contamination", "upgradient",
                      "recognized environmental condition"),
    "lease_term": ("assign this lease", "sublet", "renews automatically", "security deposit",
                   "term begins", "change of control", "quiet enjoyment"),
    "inspection_defect": ("service life", "granule loss", "heat exchanger", "cracked",
                          "broken", "deficienc", "further evaluation"),
}

# Words that move an item up the rubric, keyed to the condition names in the rules.
ESCALATION_SIGNALS = {
    "unreleased": ("unreleased", "no satisfaction"),
    "tax_lien": ("tax lien",),
    "blanket": ("blanket easement",),
    "survey_exception": ("general exception", "accurate and complete land survey"),
    "unpermitted_work": ("without a permit", "unpermitted"),
    "nonconforming": ("nonconforming",),
    "no_estoppel": ("estoppel",),
    "assignment_restricted": ("shall not assign", "without the prior written consent",
                              "change of control"),
    "structural": ("structural", "foundation"),
    "safety": ("safety hazard", "unsafe"),
    "major_system": ("roof", "hvac", "furnace", "electrical", "plumbing", "heat exchanger"),
}


class ClassifyDocsArgs(BaseModel):
    document_names: list[str] = Field(default_factory=list,
                                      description="Bundled document names, or upload paths.")
    sample_name: str = Field(default="", description="Bundled packet to read instead.")


class ExtractClausesArgs(BaseModel):
    document_name: str = Field(default="", description="One document to read.")
    document_names: list[str] = Field(default_factory=list, description="Several documents.")
    sample_name: str = Field(default="")


class CrossCheckArgs(BaseModel):
    clauses: list[dict[str, Any]] = Field(default_factory=list)
    intended_use: str = Field(default="")
    sample_name: str = Field(default="")


class SeverityArgs(BaseModel):
    items: list[dict[str, Any]] = Field(default_factory=list)
    deal_type: str = Field(default="purchase")
    sample_name: str = Field(default="")


class AttorneyQuestionsArgs(BaseModel):
    items: list[dict[str, Any]] = Field(default_factory=list)
    sample_name: str = Field(default="")


def _load_packet(sample_name: str) -> dict[str, Any]:
    ensure_samples()
    stem = (sample_name or "packet_purchase").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled packet named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


def _document_text(name: str) -> tuple[str, str]:
    """Read a bundled document or an uploaded path, returning its text and display name."""
    candidate = pathlib.Path(name)
    if candidate.is_file():
        return candidate.read_text(encoding="utf-8", errors="replace"), candidate.name
    ensure_samples()
    path = SAMPLE_DIR / f"{name.removesuffix('.txt')}.txt"
    if not path.is_file():
        raise FileNotFoundError(f"no document named '{name}'")
    return path.read_text(encoding="utf-8", errors="replace"), path.name


# When a sentence matches several categories equally, the document it came from
# decides. A restriction in a declaration is a restriction, not an easement, even
# though it mentions one.
DOCUMENT_CATEGORY_PREFERENCE = {
    "ccr": ("restriction_ccr", "zoning_permit_note", "easement"),
    "title_commitment": ("lien_encumbrance", "title_exception", "easement", "restriction_ccr"),
    "survey_notes": ("encroachment", "easement"),
    "inspection_report": ("inspection_defect",),
    "environmental_summary": ("environmental",),
    "lease": ("lease_term", "restriction_ccr"),
}


def _document_type_of(text: str) -> str:
    """Best guess at what a document is, from its wording alone."""
    lowered = text.lower()
    scores = {kind: sum(1 for marker in markers if marker in lowered)
              for kind, markers in DOCUMENT_SIGNATURES.items()}
    best = max(scores, key=lambda k: scores[k])
    return best if scores[best] else "unknown"


def _is_heading(line: str) -> bool:
    """An all-caps line is a heading, not a clause."""
    letters = [c for c in line if c.isalpha()]
    if not letters:
        return True
    return sum(1 for c in letters if c.isupper()) / len(letters) > 0.7


def _sentences(block: str) -> list[str]:
    """Substantive sentences only.

    Headings are dropped line by line, before sentences are assembled. Dropping them
    afterwards does not work: a title line glued to the mixed-case lines beneath it
    stops looking like a heading, and the page then reports a finding about its own
    letterhead.
    """
    kept = []
    for line in block.splitlines():
        text = line.strip()
        if not text or _is_heading(text):
            continue
        if text.startswith("[") and "invented" in text.lower():
            continue
        kept.append(text)

    parts = re.split(r"(?<=[.;])\s+|\n(?=\d+\.)|\n{2,}", "\n".join(kept))
    return [" ".join(p.split()) for p in parts if len(" ".join(p.split())) > 25]


@tool(args_schema=ClassifyDocsArgs)
def classify_documents(document_names: list[str] | None = None,
                       sample_name: str = "") -> dict[str, Any]:
    """Identify what each supplied document is, and name the ones a packet is missing.

    A document whose type cannot be told from its wording is reported as unknown
    rather than guessed at, because the wrong label sends the wrong clauses onward.
    """
    names = list(document_names or [])
    deal_type = "purchase"
    if not names and sample_name:
        try:
            packet = _load_packet(sample_name)
            names = list(packet.get("documents", []))
            deal_type = packet.get("deal_type", "purchase")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not names:
        return {"error": "no documents supplied", "tool_version": 1}

    classified, unknown = [], []
    for name in names:
        try:
            text, display = _document_text(name)
        except FileNotFoundError as exc:
            unknown.append({"document": name, "error": str(exc)})
            continue
        lowered = text.lower()
        scores = {kind: sum(1 for marker in markers if marker in lowered)
                  for kind, markers in DOCUMENT_SIGNATURES.items()}
        best = max(scores, key=lambda k: scores[k])
        pages = split_pages(text)
        entry = {"document": name, "file": display, "pages": len(pages),
                 "matched_markers": scores[best]}
        if scores[best] == 0:
            entry["document_type"] = "unknown"
            unknown.append(entry)
        else:
            entry["document_type"] = best
            entry["confidence"] = "high" if scores[best] >= 3 else "low"
        classified.append(entry)

    rules = load_dd_severity()
    expected = rules["expected_documents"].get(deal_type, [])
    present_types = {c.get("document_type") for c in classified}
    missing = [d for d in expected if d not in present_types]

    return {
        "deal_type": deal_type,
        "documents": classified, "document_count": len(classified),
        "unknown_documents": [u for u in unknown],
        "expected_for_deal_type": expected,
        "missing_documents": missing, "missing_count": len(missing),
        "tool_version": 1,
    }


@tool(args_schema=ExtractClausesArgs)
def extract_clauses(document_name: str = "", document_names: list[str] | None = None,
                    sample_name: str = "") -> dict[str, Any]:
    """Pull clauses out of the documents, each with its category, page and excerpt.

    Every item cites the page it came from and quotes the sentence, so a reader can
    go to the source rather than take the categorisation on trust.
    """
    names = [document_name] if document_name else list(document_names or [])
    if not names and sample_name:
        try:
            names = list(_load_packet(sample_name).get("documents", []))
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not names:
        return {"error": "no documents supplied", "tool_version": 1}

    clauses: list[dict[str, Any]] = []
    for name in names:
        try:
            text, display = _document_text(name)
        except FileNotFoundError as exc:
            clauses.append({"document": name, "error": str(exc)})
            continue
        doc_type = _document_type_of(text)
        preference = DOCUMENT_CATEGORY_PREFERENCE.get(doc_type, ())
        for page, block in split_pages(text):
            for sentence in _sentences(block):
                lowered = sentence.lower()
                scores = {
                    category: sum(1 for marker in markers if marker in lowered)
                    for category, markers in CLAUSE_SIGNATURES.items()
                }
                best = max(scores.values())
                if best == 0:
                    continue
                tied = [c for c, score in scores.items() if score == best]
                # A tie is broken by what this kind of document is mostly about.
                category = next((c for c in preference if c in tied), tied[0])
                signals = sorted({
                    condition for condition, words in ESCALATION_SIGNALS.items()
                    if any(word in lowered for word in words)})
                clauses.append({
                    "category": category, "document": name, "file": display,
                    "page": page,
                    "excerpt": sentence[:260],
                    "signals": signals,
                    "matched_markers": best,
                })

    # One finding per category per page. Several sentences on a page about the same
    # thing are one item to a reviewer, and thirty near-identical rows bury the four
    # that matter. The longest excerpt is kept and the rest are counted.
    collapsed: dict[tuple[Any, ...], dict[str, Any]] = {}
    for clause in clauses:
        if "category" not in clause:
            continue
        key = (clause["document"], clause["page"], clause["category"])
        existing = collapsed.get(key)
        if existing is None:
            collapsed[key] = {**clause, "occurrences": 1}
            continue
        existing["occurrences"] += 1
        existing["signals"] = sorted(set(existing["signals"]) | set(clause["signals"]))
        if len(clause["excerpt"]) > len(existing["excerpt"]):
            existing["excerpt"] = clause["excerpt"]
    clauses = [c for c in clauses if "category" not in c] + list(collapsed.values())

    by_category: dict[str, int] = {}
    for clause in clauses:
        if "category" in clause:
            by_category[clause["category"]] = by_category.get(clause["category"], 0) + 1

    return {
        "clauses": clauses, "clause_count": len([c for c in clauses if "category" in c]),
        "by_category": by_category,
        "documents_read": names,
        "note": ("Categories come from wording, so read the excerpt and the page before "
                 "relying on the label."),
        "tool_version": 1,
    }


@tool(args_schema=CrossCheckArgs)
def cross_check_intended_use(clauses: list[dict[str, Any]] | None = None,
                             intended_use: str = "", sample_name: str = "") -> dict[str, Any]:
    """Flag clauses that touch what the buyer says they intend to do.

    The overlap is judged on shared subject words, which is a prompt to look rather
    than a conclusion: a clause flagged here still has to be read against the plan.
    """
    items = list(clauses or [])
    use_text = intended_use
    if sample_name and (not items or not use_text):
        try:
            packet = _load_packet(sample_name)
            use_text = use_text or packet.get("intended_use", "")
            if not items:
                extracted = extract_clauses.invoke({"sample_name": sample_name})
                if "error" in extracted:
                    return extracted
                items = extracted["clauses"]
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not use_text:
        return {"error": "no intended use given, so there is nothing to compare against",
                "tool_version": 1}

    stop = {"the", "a", "an", "in", "on", "at", "to", "of", "and", "or", "for", "with",
            "build", "construct", "install", "add", "put", "my", "our", "from", "small"}
    words = {w for w in re.findall(r"[a-z]+", use_text.lower()) if w not in stop and len(w) > 2}
    # Words that mean the same thing to a reviewer reading a deed restriction.
    synonyms = {
        "garage": {"accessory structure", "detached garage", "outbuilding", "shed"},
        "rear": {"rear yard", "rear property line", "rear setback"},
        "yard": {"yard", "setback"},
        "bakery": {"commercial activity", "home occupation", "business"},
        "home": {"home occupation", "residential purposes"},
    }

    flagged = []
    for clause in items:
        if "category" not in clause:
            continue
        lowered = clause["excerpt"].lower()
        hits = sorted({w for w in words if w in lowered})
        phrase_hits = sorted({phrase for word in words
                              for phrase in synonyms.get(word, set()) if phrase in lowered})
        if not hits and not phrase_hits:
            continue
        flagged.append({
            **clause,
            "matched_words": hits, "matched_phrases": phrase_hits,
            "overlaps_intended_use": True,
            "why": (f"This clause mentions {', '.join(hits + phrase_hits)}, which appears in "
                    f"the stated intended use."),
        })

    return {
        "intended_use": use_text,
        "flagged": flagged, "flagged_count": len(flagged),
        "considered": len([c for c in items if "category" in c]),
        "note": ("An overlap is a reason to read the clause against the plan. It does not "
                 "mean the intended use is prohibited, and nothing here says whether it is "
                 "allowed."),
        "tool_version": 1,
    }


@tool(args_schema=SeverityArgs)
def severity_rubric(items: list[dict[str, Any]] | None = None, deal_type: str = "purchase",
                    sample_name: str = "") -> dict[str, Any]:
    """Sort findings by severity using the rubric, explaining every escalation.

    Severity orders a reviewer's attention. It does not say whether an item is
    curable, what it costs, or whether a transaction should proceed.
    """
    found = list(items or [])
    kind = deal_type
    overlaps: set[tuple[str, int]] = set()
    if sample_name and not found:
        try:
            packet = _load_packet(sample_name)
            kind = packet.get("deal_type", kind)
            extracted = extract_clauses.invoke({"sample_name": sample_name})
            if "error" in extracted:
                return extracted
            found = extracted["clauses"]
            crossed = cross_check_intended_use.invoke({"sample_name": sample_name})
            overlaps = {(f["category"], f["page"]) for f in crossed.get("flagged", [])}
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not found:
        return {"error": "no findings supplied", "tool_version": 1}

    rules = load_dd_severity()
    categories = rules["categories"]
    order = {"high": 3, "medium": 2, "low": 1, "needs_info": 0}
    graded = []
    for clause in found:
        category = clause.get("category")
        spec = categories.get(category)
        if spec is None:
            continue
        severity = spec["base"]
        reasons = []
        signals = set(clause.get("signals", []))
        if clause.get("overlaps_intended_use") or (category, clause.get("page")) in overlaps:
            signals.add("overlaps_intended_use")
            signals.add("prohibits_intended_use")

        for condition, description in (spec.get("escalate_to_high_when") or {}).items():
            if condition in signals:
                severity = "high"
                reasons.append(f"raised to high: {description}")
        if severity != "high":
            for condition, description in (spec.get("escalate_to_medium_when") or {}).items():
                if condition in signals and order["medium"] > order[severity]:
                    severity = "medium"
                    reasons.append(f"raised to medium: {description}")

        graded.append({
            "category": category, "label": spec["label"], "severity": severity,
            "document": clause.get("document"), "page": clause.get("page"),
            "excerpt": clause.get("excerpt", "")[:200],
            "why_it_may_matter": " ".join(spec["why_it_may_matter"].split()),
            "question_for_attorney": spec["question"],
            "escalation_reasons": reasons,
        })

    classified = classify_documents.invoke({"sample_name": sample_name}) if sample_name else {}
    for missing in classified.get("missing_documents", []):
        spec = categories["missing_document"]
        graded.append({
            "category": "missing_document", "label": spec["label"], "severity": spec["base"],
            "document": missing, "page": None, "excerpt": "",
            "why_it_may_matter": " ".join(spec["why_it_may_matter"].split()),
            "question_for_attorney": spec["question"],
            "escalation_reasons": [f"{missing} is expected for a {kind} but was not supplied"],
        })

    graded.sort(key=lambda item: (-order[item["severity"]], item["category"]))
    counts: dict[str, int] = {}
    for item in graded:
        counts[item["severity"]] = counts.get(item["severity"], 0) + 1

    levels = rules["levels_for_workflow"]
    if counts.get("high"):
        level = levels["high_severity_present"]
    elif graded:
        level = levels["any_flag_present"]
    else:
        level = levels["none_present"]

    return {
        "level": level, "deal_type": kind,
        "findings": graded, "finding_count": len(graded),
        "severity_counts": counts,
        "high_severity": [g for g in graded if g["severity"] == "high"],
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "tool_version": 1,
    }


@tool(args_schema=AttorneyQuestionsArgs)
def draft_attorney_questions(items: list[dict[str, Any]] | None = None,
                             sample_name: str = "") -> dict[str, Any]:
    """Turn the findings into a draft question list, grouped by severity.

    Questions for the client's attorney to consider and edit. Nothing here answers
    them, and none of it is legal advice.
    """
    found = list(items or [])
    if not found and sample_name:
        graded = severity_rubric.invoke({"sample_name": sample_name})
        if "error" in graded:
            return graded
        found = graded["findings"]
    if not found:
        return {"error": "no findings supplied", "tool_version": 1}

    lines = ["Questions for counsel, drafted from the documents reviewed.", ""]
    questions = []
    for severity in ("high", "medium", "low", "needs_info"):
        group = [f for f in found if f.get("severity") == severity]
        if not group:
            continue
        lines.append(f"{severity.replace('_', ' ').title()}:")
        for item in group:
            where = (f"{item.get('document', 'document')}"
                     + (f", page {item['page']}" if item.get("page") else ""))
            text = f"{item.get('label', item.get('category'))} ({where}) — " \
                   f"{item.get('question_for_attorney', '')}"
            lines.append(f"  - {text}")
            questions.append({"severity": severity, "category": item.get("category"),
                              "document": item.get("document"), "page": item.get("page"),
                              "question": item.get("question_for_attorney", "")})
        lines.append("")
    lines.append("[Draft. These are questions to raise, not conclusions, and not legal advice.]")

    return {
        "title": "Draft questions for counsel",
        "body": "\n".join(lines),
        "questions": questions, "question_count": len(questions),
        "is_draft": True,
        "note": "A draft list for an attorney to review and edit.",
        "tool_version": 1,
    }

## 4. Permit tools

Form validation, plan reading, the district standards, the
rule-by-rule check with computed ratios, the use table, and the outcome.

In [ ]:
NUMERIC_PROJECT_FIELDS = {
    "lot_area_sqft": (100, 10_000_000),
    "lot_width_ft": (5, 5000),
    "front_setback_ft": (0, 500),
    "side_setback_ft": (0, 500),
    "rear_setback_ft": (0, 500),
    "height_ft": (1, 1000),
    "stories": (1, 100),
    "gross_floor_area_sqft": (1, 10_000_000),
    "building_footprint_sqft": (1, 10_000_000),
    "dwelling_units": (0, 1000),
    "parking_spaces": (0, 10_000),
}

# Which project field each standard is measured against.
STANDARD_FIELDS = {
    "min_lot_area_sqft": "lot_area_sqft",
    "min_lot_width_ft": "lot_width_ft",
    "min_front_setback_ft": "front_setback_ft",
    "min_side_setback_ft": "side_setback_ft",
    "min_rear_setback_ft": "rear_setback_ft",
    "max_height_ft": "height_ft",
    "max_stories": "stories",
    "max_lot_coverage_pct": "lot_coverage_pct",
    "max_floor_area_ratio": "floor_area_ratio",
    "min_parking_spaces_per_unit": "parking_spaces_per_unit",
}


class ValidateProjectArgs(BaseModel):
    project: dict[str, Any] = Field(default_factory=dict, description="The submitted form.")
    sample_name: str = Field(default="", description="Bundled project to read instead.")


class ExtractPlanArgs(BaseModel):
    file_path: str = Field(default="", description="Plan text to read dimensions from.")
    project: dict[str, Any] = Field(default_factory=dict,
                                    description="The form, for conflict checking.")
    sample_name: str = Field(default="", description="Use the bundled plan notes.")


class ZoningRulesArgs(BaseModel):
    zone_district: str = Field(default="", description="R-1, R-2, C-1 or MU-1.")
    sample_name: str = Field(default="")


class DimensionalArgs(BaseModel):
    project: dict[str, Any] = Field(default_factory=dict)
    zone_district: str = Field(default="")
    sample_name: str = Field(default="")


class UsePermissionArgs(BaseModel):
    use: str = Field(default="")
    zone_district: str = Field(default="")
    sample_name: str = Field(default="")


def _load_project(project: dict[str, Any], sample_name: str) -> dict[str, Any]:
    # A named sample identifies the case under review, so it wins over an inline
    # payload. A model that half-remembers the case from an earlier step's text
    # would otherwise redirect the tool onto data it invented.
    if project and not sample_name:
        return dict(project)
    ensure_samples()
    stem = (sample_name or "project_r1_compliant").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled project named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


def _derived(project: dict[str, Any]) -> dict[str, float]:
    """Floor area ratio, lot coverage and parking per unit, computed once.

    These are the numbers a reviewer recomputes by hand, so the inputs to each are
    reported alongside the result.
    """
    out: dict[str, float] = {}
    lot = float(project.get("lot_area_sqft") or 0)
    if lot > 0:
        gross = float(project.get("gross_floor_area_sqft") or 0)
        footprint = float(project.get("building_footprint_sqft") or 0)
        out["floor_area_ratio"] = gross / lot
        out["lot_coverage_pct"] = footprint / lot * 100.0
    units = float(project.get("dwelling_units") or 0)
    if units > 0:
        out["parking_spaces_per_unit"] = float(project.get("parking_spaces") or 0) / units
    return out


@tool(args_schema=ValidateProjectArgs)
def validate_project(project: dict[str, Any] | None = None,
                     sample_name: str = "") -> dict[str, Any]:
    """Check the submitted dimensions for missing fields, bad units and impossibilities.

    A submission missing a field cannot be screened against the standard that needs
    it, so the field is named rather than assumed to be zero.
    """
    try:
        form = _load_project(project or {}, sample_name)
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    problems, missing = [], []
    for field, (low, high) in NUMERIC_PROJECT_FIELDS.items():
        raw = form.get(field)
        if raw is None or raw == "":
            missing.append(field)
            continue
        try:
            value = float(raw)
        except (TypeError, ValueError):
            problems.append(f"{field} is not a number: {raw!r}")
            continue
        if value < low or value > high:
            problems.append(f"{field} is {value:g}, outside the plausible range "
                            f"{low} to {high}")

    zone = str(form.get("zone_district", "")).upper()
    zoning = load_zoning()
    if not zone:
        missing.append("zone_district")
    elif zone not in zoning["districts"]:
        problems.append(f"zone district '{zone}' is not in the sample code",)

    if not str(form.get("use", "")).strip():
        missing.append("use")

    footprint = form.get("building_footprint_sqft")
    gross = form.get("gross_floor_area_sqft")
    try:
        if footprint and gross and float(footprint) > float(gross):
            problems.append(f"building footprint {float(footprint):g} exceeds gross floor "
                            f"area {float(gross):g}, which cannot be right")
    except (TypeError, ValueError):
        pass

    try:
        if footprint and form.get("lot_area_sqft") and \
                float(footprint) > float(form["lot_area_sqft"]):
            problems.append("building footprint exceeds the lot area")
    except (TypeError, ValueError):
        pass

    return {
        "project_name": form.get("project_name", ""),
        "zone_district": zone, "use": form.get("use", ""),
        "complete": not missing and not problems,
        "missing_fields": missing, "problems": problems,
        "derived": {k: round(v, 4) for k, v in _derived(form).items()},
        "tool_version": 1,
    }


@tool(args_schema=ExtractPlanArgs)
def extract_plan_values(file_path: str = "", project: dict[str, Any] | None = None,
                        sample_name: str = "") -> dict[str, Any]:
    """Read dimensions out of plan text and compare them against the submitted form.

    Where the plan and the form disagree, both numbers are reported. Neither is
    treated as correct: which one governs is the examiner's call.
    """
    if file_path:
        path = pathlib.Path(file_path)
    else:
        ensure_samples()
        path = SAMPLE_DIR / "plan_notes.txt"
    if not path.is_file():
        return {"error": f"no plan text at {path.name}", "tool_version": 1}

    text = path.read_text(encoding="utf-8", errors="replace")
    patterns = {
        "lot_area_sqft": r"lot area:\s*([\d,.]+)",
        "lot_width_ft": r"lot width:\s*([\d,.]+)",
        "front_setback_ft": r"front setback:\s*([\d,.]+)",
        "side_setback_ft": r"side setback:\s*([\d,.]+)",
        "rear_setback_ft": r"rear setback:\s*([\d,.]+)",
        "height_ft": r"building height:\s*([\d,.]+)",
        "stories": r"stories:\s*([\d,.]+)",
        "gross_floor_area_sqft": r"gross floor area:\s*([\d,.]+)",
        "building_footprint_sqft": r"building footprint:\s*([\d,.]+)",
    }
    found: dict[str, float] = {}
    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.I)
        if match:
            try:
                found[field] = float(match.group(1).replace(",", ""))
            except ValueError:
                continue

    form = dict(project or {})
    if not form and sample_name:
        try:
            form = _load_project({}, sample_name)
        except (FileNotFoundError, json.JSONDecodeError):
            form = {}

    conflicts = []
    for field, plan_value in found.items():
        if field not in form or form[field] in (None, ""):
            continue
        try:
            form_value = float(form[field])
        except (TypeError, ValueError):
            continue
        if abs(form_value - plan_value) > 1e-6:
            conflicts.append({"field": field, "on_form": form_value, "on_plan": plan_value,
                              "note": "The form and the plan disagree; the examiner decides "
                                      "which governs."})

    return {
        "file": path.name, "values_found": found, "value_count": len(found),
        "conflicts": conflicts, "conflict_count": len(conflicts),
        "fields_not_in_plan": sorted(set(patterns) - set(found)),
        "tool_version": 1,
    }


@tool(args_schema=ZoningRulesArgs)
def load_zoning_rules(zone_district: str = "", sample_name: str = "") -> dict[str, Any]:
    """Return the dimensional standards and use table for one district, with sections."""
    zone = zone_district.upper()
    if not zone and sample_name:
        try:
            zone = str(_load_project({}, sample_name).get("zone_district", "")).upper()
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    zoning = load_zoning()
    district = zoning["districts"].get(zone)
    if district is None:
        return {"error": f"no district '{zone}' in the sample code",
                "known": sorted(zoning["districts"]), "tool_version": 1}

    return {
        "jurisdiction": zoning["jurisdiction"],
        "zone_district": zone, "district_name": district["name"],
        "standards": {name: {"value": spec["value"], "section": spec["section"],
                             "comparison": spec["comparison"]}
                      for name, spec in district["standards"].items()},
        "uses": district["uses"],
        "code_note": " ".join(zoning["code_note"].split()),
        "tool_version": 1,
    }


@tool(args_schema=DimensionalArgs)
def check_dimensional_standards(project: dict[str, Any] | None = None, zone_district: str = "",
                                sample_name: str = "") -> dict[str, Any]:
    """Compare every dimensional standard against the proposal, one rule at a time.

    A value exactly equal to its limit passes: codes are written as "not less than"
    and "not more than", so equality is compliance rather than a near miss. Each row
    reports the required value, the proposed value and the section it came from.
    """
    try:
        form = _load_project(project or {}, sample_name)
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    zone = (zone_district or str(form.get("zone_district", ""))).upper()
    zoning = load_zoning()
    district = zoning["districts"].get(zone)
    if district is None:
        return {"error": f"no district '{zone}' in the sample code",
                "known": sorted(zoning["districts"]), "tool_version": 1}

    values = {**form, **_derived(form)}
    rows = []
    for name, spec in district["standards"].items():
        field = STANDARD_FIELDS.get(name)
        proposed = values.get(field)
        row = {"standard": name, "section": spec["section"],
               "comparison": spec["comparison"], "required": spec["value"],
               "measured_field": field}
        if proposed is None or proposed == "":
            row["result"] = "NEEDS_INFO"
            row["proposed"] = None
            row["detail"] = f"{field} was not supplied, so this standard cannot be checked."
            rows.append(row)
            continue
        try:
            proposed_value = float(proposed)
        except (TypeError, ValueError):
            row["result"] = "NEEDS_INFO"
            row["proposed"] = proposed
            row["detail"] = f"{field} is not a number."
            rows.append(row)
            continue

        limit = float(spec["value"])
        passes = proposed_value >= limit if spec["comparison"] == "at_least" \
            else proposed_value <= limit
        row["proposed"] = round(proposed_value, 4)
        row["result"] = "PASS" if passes else "FAIL"
        row["at_limit"] = abs(proposed_value - limit) < 1e-9
        word = "at least" if spec["comparison"] == "at_least" else "at most"
        row["detail"] = (f"{field} is {proposed_value:g}; {spec['section']} requires "
                         f"{word} {limit:g}.")
        rows.append(row)

    failed = [r for r in rows if r["result"] == "FAIL"]
    unknown = [r for r in rows if r["result"] == "NEEDS_INFO"]
    derived = _derived(form)
    return {
        "zone_district": zone, "district_name": district["name"],
        "rules": rows, "rule_count": len(rows),
        "failed": failed, "fail_count": len(failed),
        "needs_info": unknown, "needs_info_count": len(unknown),
        "at_limit_count": sum(1 for r in rows if r.get("at_limit")),
        "computed": {
            "floor_area_ratio": round(derived.get("floor_area_ratio", 0.0), 4),
            "floor_area_ratio_inputs": {
                "gross_floor_area_sqft": form.get("gross_floor_area_sqft"),
                "lot_area_sqft": form.get("lot_area_sqft")},
            "lot_coverage_pct": round(derived.get("lot_coverage_pct", 0.0), 4),
            "lot_coverage_inputs": {
                "building_footprint_sqft": form.get("building_footprint_sqft"),
                "lot_area_sqft": form.get("lot_area_sqft")},
            "parking_spaces_per_unit": round(derived.get("parking_spaces_per_unit", 0.0), 4),
        },
        "jurisdiction": zoning["jurisdiction"],
        "code_note": " ".join(zoning["code_note"].split()),
        "tool_version": 1,
    }


@tool(args_schema=UsePermissionArgs)
def use_permission(use: str = "", zone_district: str = "",
                   sample_name: str = "") -> dict[str, Any]:
    """Say whether a use is permitted, conditional, not permitted or unlisted here.

    An unlisted use is not a refusal: most codes have a procedure for deciding
    whether it is similar to a listed one, and that decision belongs to the examiner.
    """
    wanted, zone = use, zone_district.upper()
    if (not wanted or not zone) and sample_name:
        try:
            form = _load_project({}, sample_name)
            wanted = wanted or str(form.get("use", ""))
            zone = zone or str(form.get("zone_district", "")).upper()
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    zoning = load_zoning()
    district = zoning["districts"].get(zone)
    if district is None:
        return {"error": f"no district '{zone}' in the sample code",
                "known": sorted(zoning["districts"]), "tool_version": 1}

    status = district["uses"].get(wanted, "unlisted")
    return {
        "use": wanted, "zone_district": zone, "district_name": district["name"],
        "status": status,
        "suggested_level": zoning["use_outcomes"].get(status, "INCOMPLETE_SUBMISSION"),
        "uses_listed": sorted(district["uses"]),
        "section_prefix": district["section_prefix"],
        "note": (" ".join(zoning["unlisted_note"].split()) if status == "unlisted"
                 else "A conditional use normally requires a hearing."
                      if status == "conditional" else ""),
        "jurisdiction": zoning["jurisdiction"],
        "tool_version": 1,
    }


class PrescreenOutcomeArgs(BaseModel):
    project: dict[str, Any] = Field(default_factory=dict)
    sample_name: str = Field(default="")


@tool(args_schema=PrescreenOutcomeArgs)
def prescreen_outcome(project: dict[str, Any] | None = None,
                      sample_name: str = "") -> dict[str, Any]:
    """Combine completeness, the dimensional results and the use table into one level.

    Every input is measured rather than supplied, so the level always rests on what
    the other tools found. A pre-screen tells an examiner where to look; the examiner
    decides, and nothing here approves, denies or issues anything.
    """
    try:
        form = _load_project(project or {}, sample_name)
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    validated = validate_project.invoke({"project": form})
    dimensional = check_dimensional_standards.invoke({"project": form})
    if "error" in dimensional:
        return dimensional
    permission = use_permission.invoke({"use": str(form.get("use", "")),
                                        "zone_district": str(form.get("zone_district", ""))})

    reasons: list[str] = []
    if not validated["complete"]:
        level = "INCOMPLETE_SUBMISSION"
        for field in validated["missing_fields"]:
            reasons.append(f"{field} was not supplied")
        reasons.extend(validated["problems"])
    elif dimensional["needs_info_count"]:
        level = "INCOMPLETE_SUBMISSION"
        reasons.append(f"{dimensional['needs_info_count']} standard(s) could not be checked")
    elif permission.get("status") == "not_permitted":
        level = "CORRECTIONS_LIKELY"
        reasons.append(f"{permission['use']} is not permitted in {permission['zone_district']}")
    elif dimensional["fail_count"]:
        level = "CORRECTIONS_LIKELY"
        for row in dimensional["failed"]:
            reasons.append(f"{row['standard']} fails {row['section']}: {row['detail']}")
    elif permission.get("status") == "conditional":
        level = "HEARING_LIKELY"
        reasons.append(f"{permission['use']} is a conditional use in "
                       f"{permission['zone_district']}, which normally requires a hearing")
    elif permission.get("status") == "unlisted":
        level = "INCOMPLETE_SUBMISSION"
        reasons.append(f"{permission['use']} is not listed in {permission['zone_district']}")
    else:
        level = "READY_FOR_EXAMINER"
        reasons.append("every dimensional standard passes and the use is permitted")

    return {
        "level": level, "reasons": reasons,
        "use_status": permission.get("status"),
        "fail_count": dimensional["fail_count"],
        "needs_info_count": dimensional["needs_info_count"],
        "missing_field_count": len(validated["missing_fields"]),
        "zone_district": dimensional["zone_district"],
        "jurisdiction": dimensional["jurisdiction"],
        "note": ("A pre-screen against a fictional sample code. It does not approve, deny "
                 "or issue a permit, and an examiner decides."),
        "tool_version": 1,
    }

## 5. Dispute tools

Timeline assembly, clause matching, each party's position,
placeholder notice dates, tone flags, and the outcome.

In [ ]:
# A dated line in correspondence, e.g. "2026-03-24 - HOA to Owner: ...".
CORRESPONDENCE_LINE = re.compile(
    r"^\s*(\d{4}-\d{2}-\d{2})\s*[-–]\s*(.+?)\s*:\s*(.+)$")

REQUEST_MARKERS = ("i am requesting", "i request", "please send", "please return",
                   "i am asking", "requesting a hearing", "please put", "please restore",
                   "asking you to")
POSITION_MARKERS = ("i submitted", "i did not", "i have not", "there was", "we have",
                    "i moved out", "the association says", "the manager has",
                    "i went ahead", "we are still")


class TimelineArgs(BaseModel):
    correspondence: str = Field(default="", description="Dated correspondence, one per line.")
    sample_name: str = Field(default="", description="Bundled dispute to read instead.")


class MatchClausesArgs(BaseModel):
    issue: str = Field(default="", description="What the dispute is about.")
    document_names: list[str] = Field(default_factory=list)
    sample_name: str = Field(default="")


class PositionsArgs(BaseModel):
    texts: list[str] = Field(default_factory=list)
    sample_name: str = Field(default="")


class NoticeDeadlinesArgs(BaseModel):
    dispute_type: str = Field(default="")
    events: list[dict[str, Any]] = Field(default_factory=list)
    jurisdiction: str = Field(default="")
    sample_name: str = Field(default="")


class ToneFlagsArgs(BaseModel):
    texts: list[str] = Field(default_factory=list)
    sample_name: str = Field(default="")


def _load_dispute(sample_name: str) -> dict[str, Any]:
    ensure_samples()
    stem = (sample_name or "dispute_hoa_fence").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled dispute named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


def _dispute_texts(dispute: dict[str, Any]) -> list[str]:
    return [t for t in (dispute.get("complaint_text", ""),
                        dispute.get("correspondence", "")) if t]


@tool(args_schema=TimelineArgs)
def build_timeline(correspondence: str = "", sample_name: str = "") -> dict[str, Any]:
    """Assemble the dated events from correspondence, oldest first.

    Only what is dated in the source appears here. Gaps between events are reported
    because in a dispute the silences are often the point.
    """
    body = correspondence
    if not body and sample_name:
        try:
            body = _load_dispute(sample_name).get("correspondence", "")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not body:
        return {"error": "no correspondence supplied", "tool_version": 1}

    events = []
    for line in body.splitlines():
        match = CORRESPONDENCE_LINE.match(line)
        if not match:
            continue
        raw_date, party, text = match.groups()
        when = _date(raw_date)
        if when is None:
            continue
        events.append({"date": when.isoformat(), "from": party.strip(),
                       "summary": " ".join(text.split())[:220]})
    events.sort(key=lambda e: e["date"])

    gaps = []
    for earlier, later in zip(events, events[1:], strict=False):
        days = (_date(later["date"]) - _date(earlier["date"])).days
        if days >= 14:
            gaps.append({"from": earlier["date"], "to": later["date"], "days": days,
                         "note": "A gap this long is worth asking both parties about."})

    first, last = (events[0]["date"], events[-1]["date"]) if events else ("", "")
    return {
        "events": events, "event_count": len(events),
        "first_event": first, "last_event": last,
        "span_days": (_date(last) - _date(first)).days if events else 0,
        "gaps": gaps, "gap_count": len(gaps),
        "note": "Only dated items in the supplied correspondence appear here.",
        "tool_version": 1,
    }


@tool(args_schema=MatchClausesArgs)
def match_clauses(issue: str = "", document_names: list[str] | None = None,
                  sample_name: str = "") -> dict[str, Any]:
    """Find the lease or association clauses that bear on the issue, with page references.

    Paraphrases where it can and quotes only the sentence it found, so the reader
    goes to the page rather than relying on the summary.
    """
    names = list(document_names or [])
    subject = issue
    if sample_name and (not names or not subject):
        try:
            dispute = _load_dispute(sample_name)
            names = names or list(dispute.get("documents", []))
            subject = subject or dispute.get("dispute_type", "")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not names:
        return {"error": "no documents supplied", "tool_version": 1}

    topics = {
        "hoa_rule": ("architectural", "approved in writing", "accessory structure", "fence",
                     "committee"),
        "lease_violation": ("tenant shall", "landlord", "assign", "sublet", "breach"),
        "deposit": ("security deposit", "deposit"),
        "noise": ("noise", "quiet enjoyment", "disturb"),
        "maintenance": ("repair", "maintain", "habitab"),
        "eviction_related": ("terminate", "possession", "notice"),
    }
    markers = topics.get(subject, ())

    matches = []
    for name in names:
        try:
            text, display = _document_text(name)
        except FileNotFoundError as exc:
            matches.append({"document": name, "error": str(exc)})
            continue
        for page, block in split_pages(text):
            for sentence in _sentences(block):
                lowered = sentence.lower()
                hit = [m for m in markers if m in lowered]
                if not hit:
                    continue
                matches.append({
                    "document": name, "file": display, "page": page,
                    "matched_terms": hit,
                    "quote": sentence[:200],
                    "relevance": f"Mentions {', '.join(hit)}, which relates to a "
                                 f"{subject.replace('_', ' ')} dispute.",
                })

    seen, unique = set(), []
    for match in matches:
        key = (match.get("document"), match.get("page"), match.get("quote", "")[:60])
        if key in seen:
            continue
        seen.add(key)
        unique.append(match)

    return {
        "issue": subject, "clauses": unique, "clause_count": len(unique),
        "documents_read": names,
        "note": ("Clauses are matched on wording. Whether one applies to these facts is "
                 "for the reader, and local law may override what a document says."),
        "tool_version": 1,
    }


@tool(args_schema=PositionsArgs)
def positions_extractor(texts: list[str] | None = None,
                        sample_name: str = "") -> dict[str, Any]:
    """Set out what each party says happened and what they are asking for.

    Stated positions are reported as stated, attributed to whoever said them, and
    not weighed against each other.
    """
    bodies = list(texts or [])
    parties: list[dict[str, str]] = []
    if sample_name and not bodies:
        try:
            dispute = _load_dispute(sample_name)
            bodies = _dispute_texts(dispute)
            parties = dispute.get("parties", [])
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not bodies:
        return {"error": "no text supplied", "tool_version": 1}

    by_party: dict[str, dict[str, list[str]]] = {}
    for body in bodies:
        for line in body.splitlines():
            match = CORRESPONDENCE_LINE.match(line)
            speaker, content = (match.group(2).strip(), match.group(3)) if match \
                else ("complainant", line)
            content = " ".join(content.split())
            if len(content) < 20:
                continue
            lowered = content.lower()
            entry = by_party.setdefault(speaker, {"statements": [], "requests": []})
            if any(marker in lowered for marker in REQUEST_MARKERS):
                entry["requests"].append(content[:220])
            elif any(marker in lowered for marker in POSITION_MARKERS):
                entry["statements"].append(content[:220])

    positions = [{"party": speaker, "statements": entry["statements"][:4],
                  "requests": entry["requests"][:4],
                  "statement_count": len(entry["statements"]),
                  "request_count": len(entry["requests"])}
                 for speaker, entry in by_party.items()]

    return {
        "parties_on_record": parties,
        "positions": positions, "party_count": len(positions),
        "note": ("Positions are what each side says, reported without weighing them. "
                 "Nothing here decides who is right."),
        "tool_version": 1,
    }


@tool(args_schema=NoticeDeadlinesArgs)
def notice_deadlines(dispute_type: str = "", events: list[dict[str, Any]] | None = None,
                     jurisdiction: str = "", sample_name: str = "") -> dict[str, Any]:
    """Compute dates from placeholder notice periods, labelled as placeholders.

    The periods are invented for this project. Every date below is arithmetic from a
    placeholder, not a deadline: the real period comes from the agreement and local
    law, and both vary widely.
    """
    kind = dispute_type
    where = jurisdiction
    timeline = list(events or [])
    if sample_name and (not kind or not timeline):
        try:
            dispute = _load_dispute(sample_name)
            kind = kind or dispute.get("dispute_type", "")
            where = where or dispute.get("jurisdiction", "")
            if not timeline:
                built = build_timeline.invoke({"sample_name": sample_name})
                if "error" in built:
                    return built
                timeline = built["events"]
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    rules = load_notice_periods()
    periods = rules["periods"].get(kind)
    if periods is None:
        return {"error": f"no placeholder periods for dispute type '{kind}'",
                "known": sorted(rules["periods"]), "tool_version": 1}

    anchor = _date(timeline[-1]["date"]) if timeline else TODAY
    computed = []
    for name, value in periods.items():
        if not isinstance(value, int):
            continue
        due = anchor + dt.timedelta(days=value)
        computed.append({
            "period": name, "placeholder_days": value,
            "measured_from": anchor.isoformat(), "date": due.isoformat(),
            "days_remaining": (due - TODAY).days,
            "label": "placeholder — confirm local law",
        })

    return {
        "dispute_type": kind, "jurisdiction": where or "not stated",
        "anchor_event": anchor.isoformat(),
        "computed_dates": computed, "date_count": len(computed),
        "period_note": periods.get("note", ""),
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "tool_version": 1,
    }


@tool(args_schema=ToneFlagsArgs)
def tone_flags(texts: list[str] | None = None, sample_name: str = "") -> dict[str, Any]:
    """Flag threatening or discriminatory wording, and say what it should trigger.

    Matching on phrasing is crude and catches innocent wording, so a flag is a reason
    for a person to read the correspondence. Where a flag suggests a threat or
    discriminatory language, the recommendation is counsel or a fair-housing review
    rather than a summary, because that is not a judgement this should make.
    """
    bodies = list(texts or [])
    kind = ""
    if sample_name and not bodies:
        try:
            dispute = _load_dispute(sample_name)
            bodies = _dispute_texts(dispute)
            kind = dispute.get("dispute_type", "")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}
    if not bodies:
        return {"error": "no text supplied", "tool_version": 1}

    rules = load_notice_periods()
    markers = rules["tone_markers"]
    blob = "\n".join(bodies)
    lowered = blob.lower()

    found: dict[str, list[dict[str, str]]] = {}
    for kind_name, phrases in markers.items():
        for phrase in phrases:
            index = lowered.find(phrase)
            if index < 0:
                continue
            start = max(0, index - 70)
            found.setdefault(kind_name, []).append({
                "phrase": phrase,
                "context": " ".join(blob[start:index + len(phrase) + 70].split()),
            })

    escalate = rules["always_escalate"]
    triggered = [k for k in found if k in escalate["signal_kinds"]]
    if kind in escalate["dispute_types"]:
        triggered.append(f"dispute type {kind}")

    return {
        "flags": found,
        "flag_kinds": sorted(found),
        "threat_count": len(found.get("threat", [])),
        "discrimination_count": len(found.get("discrimination", [])),
        "escalation_count": len(found.get("escalation", [])),
        "escalation_triggered": bool(triggered),
        "escalation_reasons": triggered,
        "recommended_level": ("ESCALATE_TO_COUNSEL" if triggered else ""),
        "escalation_note": " ".join(escalate["reason"].split()),
        "interpretation_note": (
            "Wording matches, not findings. Frustrated language is common in a genuine "
            "dispute and is not itself misconduct; a person reads the correspondence."),
        "tool_version": 1,
    }


class DisputeOutcomeArgs(BaseModel):
    dispute_type: str = Field(default="")
    sample_name: str = Field(default="")


@tool(args_schema=DisputeOutcomeArgs)
def dispute_outcome(dispute_type: str = "", sample_name: str = "") -> dict[str, Any]:
    """Decide the level from the tone flags, the dispute type and what is on record.

    Eviction, threats and apparent discriminatory language always go to counsel,
    whatever else the record shows. Otherwise the question is only whether enough is
    documented to summarise, never who is right.
    """
    kind = dispute_type
    if sample_name and not kind:
        try:
            kind = _load_dispute(sample_name).get("dispute_type", "")
        except (FileNotFoundError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    flags = tone_flags.invoke({"sample_name": sample_name}) if sample_name else {}
    timeline = build_timeline.invoke({"sample_name": sample_name}) if sample_name else {}
    positions = positions_extractor.invoke({"sample_name": sample_name}) if sample_name else {}

    rules = load_notice_periods()
    escalate = rules["always_escalate"]
    reasons: list[str] = []

    forced = list(flags.get("escalation_reasons", []))
    if kind in escalate["dispute_types"] and f"dispute type {kind}" not in forced:
        forced.append(f"dispute type {kind}")

    if forced:
        level = "ESCALATE_TO_COUNSEL"
        for reason in forced:
            reasons.append(f"escalation required: {reason}")
        reasons.append(" ".join(escalate["reason"].split()))
    elif timeline.get("event_count", 0) < 2 or positions.get("party_count", 0) < 2:
        level = "NEEDS_MORE_INFORMATION"
        reasons.append("too little is on record to summarise both sides fairly")
    else:
        level = "MEDIATION_CANDIDATE"
        reasons.append(f"{timeline.get('event_count', 0)} dated events and "
                       f"{positions.get('party_count', 0)} parties are on record, with no "
                       f"threat or discrimination signal")

    return {
        "level": level, "dispute_type": kind, "reasons": reasons,
        "escalation_reasons": forced,
        "event_count": timeline.get("event_count", 0),
        "party_count": positions.get("party_count", 0),
        "note": ("A summary for a person to act on. Courts and authorised bodies decide "
                 "evictions, and nothing here decides any dispute."),
        "tool_version": 1,
    }

## 6. Prompt addenda and workflows

Sector guidance and the three workflow definitions.

In [ ]:
ADDENDA = {
    "coordinator": (
        "Identify and explain issues; never conclude whether an issue kills a deal, is legal, "
        "or is compliant. Point each item to the document and the page it came from. Local law "
        "varies, sample rules here belong to a fictional jurisdiction, and a licensed attorney, "
        "permit examiner or property manager decides."
    ),
    "researcher": (
        "Quote at most one short phrase per clause; otherwise paraphrase with a page reference. "
        "Anything you cannot point to a page for is marked as needing verification."
    ),
    "analyst": (
        "Report the required value, the proposed value and the section for every standard you "
        "check, so a reviewer can redo the arithmetic. Severity orders attention and says "
        "nothing about whether an issue is curable."
    ),
    "executor": (
        "Name the documents that were not supplied rather than working around them. A clause "
        "you could not find a page for does not go in the table."
    ),
}


class RedFlagInputs(BaseModel):
    property_address: str = Field(default="", description="The property under review.")
    deal_type: str = Field(default="purchase", description="purchase, lease or refinance.")
    intended_use: str = Field(default="", description="What the buyer intends to do.")
    sample_name: str = Field(default="packet_purchase", description="Bundled document packet.")


class PermitInputs(BaseModel):
    zone_district: str = Field(default="R-1", description="R-1, R-2, C-1 or MU-1.")
    use: str = Field(default="single_family_dwelling")
    lot_area_sqft: float = Field(default=0.0)
    lot_width_ft: float = Field(default=0.0)
    front_setback_ft: float = Field(default=0.0)
    side_setback_ft: float = Field(default=0.0)
    rear_setback_ft: float = Field(default=0.0)
    height_ft: float = Field(default=0.0)
    stories: int = Field(default=0)
    gross_floor_area_sqft: float = Field(default=0.0)
    building_footprint_sqft: float = Field(default=0.0)
    dwelling_units: int = Field(default=0)
    parking_spaces: int = Field(default=0)
    sample_name: str = Field(default="project_r1_setback_violation",
                             description="Bundled project instead of the form.")


class DisputeInputs(BaseModel):
    dispute_type: str = Field(default="noise",
                              description="lease_violation, hoa_rule, maintenance, deposit, "
                                          "noise or eviction_related.")
    complaint_text: str = Field(default="")
    correspondence: str = Field(default="")
    jurisdiction: str = Field(default="", description="Optional; affects nothing but labelling.")
    sample_name: str = Field(default="dispute_noise_escalating")


RED_FLAG_PLAN = Plan(
    objective="Assemble a red-flag memo from a due-diligence packet for attorney review.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Identify each supplied document and name anything a packet of "
                             "this kind would normally include but does not.",
                 tool_hints=["classify_documents"], expected_output="document inventory"),
        PlanStep(id="s2", agent="executor",
                 instruction="Extract the clauses from each document with the page and the "
                             "sentence they came from.",
                 tool_hints=["extract_clauses"], depends_on=["s1"],
                 expected_output="clauses with pages"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Flag clauses touching the stated intended use, then grade "
                             "everything against the severity rubric.",
                 tool_hints=["cross_check_intended_use", "severity_rubric"], depends_on=["s2"],
                 expected_output="graded findings"),
        PlanStep(id="s4", agent="researcher",
                 instruction="Draft the question list for counsel, grouped by severity.",
                 tool_hints=["draft_attorney_questions"], depends_on=["s3"],
                 expected_output="draft questions"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find explanations of these issue types and the usual follow-up "
                             "questions.",
                 tool_hints=["search_knowledge"], expected_output="cited context"),
    ],
)

PERMIT_PLAN = Plan(
    objective="Pre-screen a project against the district's standards for an examiner.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Validate the submitted dimensions and load the standards for "
                             "the district, with their section numbers.",
                 tool_hints=["validate_project", "load_zoning_rules"],
                 expected_output="validated form and standards"),
        PlanStep(id="s2", agent="executor",
                 instruction="Read the plan text and report anything that disagrees with "
                             "the form.",
                 tool_hints=["extract_plan_values"], depends_on=["s1"],
                 expected_output="plan values and conflicts"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Check every dimensional standard and the use table, reporting "
                             "required against proposed for each.",
                 tool_hints=["check_dimensional_standards", "use_permission"],
                 depends_on=["s1"], expected_output="rule by rule results"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Give the pre-screen outcome and the reasons behind it.",
                 tool_hints=["prescreen_outcome"], depends_on=["s3"],
                 expected_output="a level with reasons"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find the code sections to cite alongside each result.",
                 tool_hints=["search_knowledge"], expected_output="cited sections"),
    ],
)

DISPUTE_PLAN = Plan(
    objective="Summarise a tenant or association dispute neutrally for whoever handles it.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Build the dated timeline from the correspondence and set out "
                             "what each party says and asks for.",
                 tool_hints=["build_timeline", "positions_extractor"],
                 expected_output="timeline and positions"),
        PlanStep(id="s2", agent="researcher",
                 instruction="Find the lease or association clauses that bear on the issue, "
                             "with page references.",
                 tool_hints=["match_clauses"], depends_on=["s1"],
                 expected_output="relevant clauses"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Compute the placeholder notice dates and flag threatening or "
                             "discriminatory wording.",
                 tool_hints=["notice_deadlines", "tone_flags"], depends_on=["s1"],
                 expected_output="dates and tone flags"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Give the outcome level and say plainly what forced it.",
                 tool_hints=["dispute_outcome"], depends_on=["s3"],
                 expected_output="a level with reasons"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find mediation practice and comparable past cases.",
                 tool_hints=["search_knowledge"], expected_output="cited practice"),
    ],
)

# Demo-mode scripts. Arguments are fixed, which is why every tool accepts a sample
# name or falls back to what the previous step loaded.
RED_FLAG_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "classify_documents",
                            "args": {"sample_name": "packet_purchase"}}]}, "identified"],
    "s2": [{"tool_calls": [{"name": "extract_clauses",
                            "args": {"sample_name": "packet_purchase"}}]}, "extracted"],
    "s3": [{"tool_calls": [
        {"name": "cross_check_intended_use", "args": {"sample_name": "packet_purchase"}},
        {"name": "severity_rubric", "args": {"sample_name": "packet_purchase"}}]}, "graded"],
    "s4": [{"tool_calls": [{"name": "draft_attorney_questions",
                            "args": {"sample_name": "packet_purchase"}}]}, "drafted"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "title exception easement encroachment due diligence review", "k": 4,
        "include_cases": True}}]}, "context gathered"],
}

PERMIT_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "validate_project", "args": {"sample_name": "project_r1_setback_violation"}},
        {"name": "load_zoning_rules", "args": {"zone_district": "R-1"}}]}, "validated"],
    "s2": [{"tool_calls": [{"name": "extract_plan_values",
                            "args": {"sample_name": "project_r1_setback_violation"}}]},
        "plan read"],
    "s3": [{"tool_calls": [
        {"name": "check_dimensional_standards",
         "args": {"sample_name": "project_r1_setback_violation"}},
        {"name": "use_permission", "args": {"use": "single_family_dwelling",
                                            "zone_district": "R-1"}}]}, "checked"],
    "s4": [{"tool_calls": [{"name": "prescreen_outcome",
                            "args": {"sample_name": "project_r1_setback_violation"}}]},
        "screened"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "zoning setback floor area ratio lot coverage standards", "k": 4}}]},
        "sections found"],
}

DISPUTE_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "build_timeline", "args": {"sample_name": "dispute_noise_escalating"}},
        {"name": "positions_extractor",
         "args": {"sample_name": "dispute_noise_escalating"}}]}, "assembled"],
    "s2": [{"tool_calls": [{"name": "match_clauses",
                            "args": {"sample_name": "dispute_noise_escalating"}}]}, "matched"],
    "s3": [{"tool_calls": [
        {"name": "notice_deadlines", "args": {"sample_name": "dispute_noise_escalating"}},
        {"name": "tone_flags", "args": {"sample_name": "dispute_noise_escalating"}}]},
        "flagged"],
    "s4": [{"tool_calls": [{"name": "dispute_outcome",
                            "args": {"sample_name": "dispute_noise_escalating"}}]}, "decided"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "mediation practice tenant association dispute escalation", "k": 4,
        "include_cases": True}}]}, "practice found"],
}

RED_FLAG_WORKFLOW = WorkflowSpec(
    id="realestate.dd_redflags",
    name="Due-diligence red-flag memo",
    description="Read a diligence packet and list what an attorney should look at, with pages.",
    input_schema=RedFlagInputs,
    accepted_uploads=[".pdf", ".docx", ".txt"],
    step_template="1. identify documents  2. extract clauses  3. cross-check use and grade  "
                  "4. draft questions  5. gather context",
    default_plan=RED_FLAG_PLAN,
    fake_script={"steps": RED_FLAG_SCRIPT},
    level_vocab=["NO_MAJOR_FLAGS_FOUND", "FLAGS_FOR_ATTORNEY_REVIEW", "HIGH_SEVERITY_FLAGS"],
    forbidden_phrases=[r"\bdeal[- ]killer\b", r"\b(?:is|are) (?:legal|illegal|compliant)\b",
                       r"\bsafe to close\b", r"\bclear to close\b"],
    sample_name="packet_purchase",
    example_request="Review this diligence packet before we go to the attorney.",
)

PERMIT_WORKFLOW = WorkflowSpec(
    id="realestate.permit_prescreen",
    name="Permit and zoning pre-screen",
    description="Check a project against the district's dimensional standards and use table.",
    input_schema=PermitInputs,
    accepted_uploads=[".pdf", ".txt"],
    step_template="1. validate and load standards  2. read the plan  3. check rules and use  "
                  "4. outcome  5. gather sections",
    default_plan=PERMIT_PLAN,
    fake_script={"steps": PERMIT_SCRIPT},
    level_vocab=["READY_FOR_EXAMINER", "CORRECTIONS_LIKELY", "HEARING_LIKELY",
                 "INCOMPLETE_SUBMISSION"],
    forbidden_phrases=[r"\bpermit (?:is|has been) (?:approved|denied|issued)\b",
                       r"\bvariance (?:is )?granted\b", r"\bfully compliant\b"],
    sample_name="project_r1_setback_violation",
    example_request="Pre-screen this addition against the R-1 standards.",
)

DISPUTE_WORKFLOW = WorkflowSpec(
    id="realestate.dispute_summary",
    name="Tenant and association dispute summary",
    description="Summarise a dispute neutrally, with a timeline, clauses and next steps.",
    input_schema=DisputeInputs,
    accepted_uploads=[".txt", ".pdf", ".docx"],
    step_template="1. timeline and positions  2. clauses  3. dates and tone  4. outcome  "
                  "5. gather practice",
    default_plan=DISPUTE_PLAN,
    fake_script={"steps": DISPUTE_SCRIPT},
    level_vocab=["MEDIATION_CANDIDATE", "NEEDS_MORE_INFORMATION", "ESCALATE_TO_COUNSEL"],
    forbidden_phrases=[r"\btenant (?:is|was) (?:evicted|at fault)\b",
                       r"\bowner (?:is|was) at fault\b",
                       r"\byou (?:must|should) evict\b",
                       r"\blandlord (?:is|was) at fault\b"],
    sample_name="dispute_noise_escalating",
    example_request="Summarise this dispute and tell me what the next step should be.",
)

## 7. Sector pack

Tool allowlists per role, and registration.

In [ ]:
SECTOR_PACK = SectorPack(
    **sector_identity(SECTOR_ID),
    tools=[
        ToolSpec(tool=classify_documents, roles=["executor"]),
        ToolSpec(tool=extract_clauses, roles=["executor"]),
        ToolSpec(tool=cross_check_intended_use, roles=["analyst"]),
        ToolSpec(tool=severity_rubric, roles=["analyst"]),
        ToolSpec(tool=draft_attorney_questions, roles=["researcher"]),
        ToolSpec(tool=validate_project, roles=["executor"]),
        ToolSpec(tool=extract_plan_values, roles=["executor"]),
        ToolSpec(tool=load_zoning_rules, roles=["executor"]),
        ToolSpec(tool=check_dimensional_standards, roles=["analyst"]),
        ToolSpec(tool=use_permission, roles=["analyst"]),
        ToolSpec(tool=prescreen_outcome, roles=["analyst"]),
        ToolSpec(tool=build_timeline, roles=["executor"]),
        ToolSpec(tool=positions_extractor, roles=["executor"]),
        ToolSpec(tool=match_clauses, roles=["researcher"]),
        ToolSpec(tool=notice_deadlines, roles=["analyst"]),
        ToolSpec(tool=tone_flags, roles=["analyst"]),
        ToolSpec(tool=dispute_outcome, roles=["analyst"]),
    ],
    workflows=[RED_FLAG_WORKFLOW, PERMIT_WORKFLOW, DISPUTE_WORKFLOW],
    addenda=ADDENDA,
    ensure_samples=ensure_samples,
)

register_sector(SECTOR_PACK)

## Demo

Examples only; this cell is dropped from the built module.

In [ ]:
# Run every workflow offline and show what the reviewer would see.
import asyncio

ensure_samples()
for workflow in SECTOR_PACK.workflows:
    print(f"--- {workflow.id} ---")
    run_id = asyncio.run(start_run("realestate", workflow.id, workflow.example_request, {}))
    view = asyncio.run(get_run(run_id))
    print(view.status, view.brief["recommendation_level"] if view.brief else "no brief")

## Build check

Confirms the notebook reached the generated module.

In [ ]:
def hello() -> str:
    """Return this module's name, so the build pipeline can be checked end to end."""
    return "automatron_realestate"